In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/intern.task2
!pwd
!ls

In [ ]:
import pandas as pd, numpy as np

In [ ]:
products = ["Laptop","Phone","Tablet","Headphones","Monitor","Keyboard","Mouse","Webcam"]
prices   = [900, 700, 450, 120, 300, 60, 25, 80]
regions  = ["North","South","East","West"]

In [ ]:
rows = []
for d in pd.date_range("2026-09-21", "2026-10-18"):   # 4 full Mon-Sun weeks
    for _ in range(np.random.randint(15, 30)):
        i = np.random.randint(len(products))
        units = np.random.randint(1, 6)
        rows.append([d.date(), products[i], np.random.choice(regions),
                     units, prices[i], units * prices[i]])

In [ ]:
pd.DataFrame(rows, columns=["date","product","region","units","unit_price","revenue"]) \
  .to_csv("sales.csv", index=False)

print("sales.csv created")
pd.read_csv("sales.csv").head()

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("sales.csv", parse_dates=["date"])

In [ ]:
df["week_start"] = df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D")


In [ ]:
weeks = sorted(df["week_start"].unique())
this_w, last_w = weeks[-1], weeks[-2]
cur  = df[df["week_start"] == this_w]
prev = df[df["week_start"] == last_w]

In [ ]:
top5      = cur.groupby("product")["revenue"].sum().nlargest(5)
reg_cur   = cur.groupby("region")["revenue"].sum()
reg_prev  = prev.groupby("region")["revenue"].sum()
reg_wow   = ((reg_cur - reg_prev) / reg_prev * 100).round(1)
total_cur, total_prev = cur["revenue"].sum(), prev["revenue"].sum()
total_wow = (total_cur - total_prev) / total_prev * 100

In [ ]:
lines = [
    "Subject: Weekly Sales Report",
    "",
    f"Week starting {pd.Timestamp(this_w).date()}",
    f"Total revenue: ${total_cur:,.0f} ({total_wow:+.1f}% vs last week)",
    "",
    "TOP 5 PRODUCTS",
]
for i, (p, r) in enumerate(top5.items(), 1):
    lines.append(f"  {i}. {p}: ${r:,.0f}")


In [ ]:
lines += ["", "REVENUE BY REGION (week-over-week)"]
for region, rev in reg_cur.items():
    lines.append(f"  {region}: ${rev:,.0f} ({reg_wow[region]:+.1f}%)")

message = "\n".join(lines)
print(message)

with open("sample_output.txt", "w") as f:
    f.write(message)
print("\nSaved to sample_output.txt")

In [27]:
import smtplib, getpass
from email.mime.text import MIMEText

sender   = input("Your Gmail address: ")
password = getpass.getpass("Gmail app password: ")
receiver = input("Send to (can be yourself): ")

body = message.split("\n", 2)[2]
msg = MIMEText(body)
msg["Subject"] = "Weekly Sales Report"
msg["From"], msg["To"] = sender, receiver

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
    s.login(sender, password)
    s.send_message(msg)
print("Email sent")

Your Gmail address: wemuteygilbert@gmail.com


KeyboardInterrupt: Interrupted by user